# Adapter trimming — ERP003950

Mouse IgG heavy-chain amplicon dataset `ERP003950` (`Greiff 2014`).

- Input: `/data/user/epishkin/raw/ERP003950/*.fastq.gz`
- Output: `/data/user/epishkin/results/ERP003950/trimmed/fastq`
- This notebook handles **adapter/quality trimming only**.
- FR1/CH1 technical primers are trimmed in the separate `primer_trim.ipynb` notebook.

Design choice: after smoke-test autodetection on real `ERP003950` reads, explicit Illumina/TruSeq-style adapters are hardcoded here and trimming is done with `cutadapt` only, by analogy with the human notebook. No primer trimming is performed at this stage.


In [ ]:
import os, sys, sysconfig, shutil, subprocess
_CONDA_ENV = '/opt/conda/envs/bcr_env'
os.environ['PATH'] = _CONDA_ENV + '/bin:' + os.environ.get('PATH', '')
os.environ['PYTHONNOUSERSITE'] = '1'
sys.path[:] = [p for p in sys.path if '/data/user/epishkin/.local' not in p]
for _site in [_CONDA_ENV + '/lib/python3.11/site-packages', sysconfig.get_path('purelib')]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)
os.environ['HOME'] = '/data/user/epishkin'
os.environ['XDG_CONFIG_HOME'] = '/data/user/epishkin/.config'
os.makedirs(os.environ['XDG_CONFIG_HOME'], exist_ok=True)


In [ ]:
from pathlib import Path

TRIM_QUALITY = '0,30'
MIN_LENGTH = 250
ADAPTER_TIMES = 2
ADAPTER_MIN_OVERLAP = 10

# Smoke-test autodetect on ERR346596 (2026-07-26)
ILLUMINA_ADAPTER_R1 = 'AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'
ILLUMINA_ADAPTER_R2 = 'GATCGGAAGAGCACACGTCTGAACTCCAGTCAC'

def run_adapter_trim(volume, dataset, force=False):
    vol = Path(volume)
    src = vol / 'raw' / dataset
    base = vol / 'results' / dataset / 'trimmed'
    out = base / 'fastq'

    if not src.is_dir():
        raise FileNotFoundError(f'Raw FASTQ dir not found: {src}')

    if force and base.exists():
        shutil.rmtree(base)

    out.mkdir(parents=True, exist_ok=True)

    pairs = sorted(set(f.name.rsplit('_', 1)[0] for f in src.glob('*.fastq.gz')))
    print(f'[adapter_trim] {dataset}: {len(pairs)} pairs')
    print(f'[adapter_trim] quality={TRIM_QUALITY} minlen={MIN_LENGTH} times={ADAPTER_TIMES} overlap={ADAPTER_MIN_OVERLAP}')

    for bn in pairs:
        r1 = src / f'{bn}_1.fastq.gz'
        r2 = src / f'{bn}_2.fastq.gz'
        if not r1.exists() or not r2.exists():
            continue

        o1 = out / f'{bn}_1.trim.fastq.gz'
        o2 = out / f'{bn}_2.trim.fastq.gz'

        if o1.exists() and o2.exists():
            print(f'  [{bn}] already done, skip')
            continue

        ca = ['cutadapt', '--quality-cutoff', TRIM_QUALITY, '--compression-level', '1', '-m', str(MIN_LENGTH), '--times', str(ADAPTER_TIMES), '-O', str(ADAPTER_MIN_OVERLAP)]
        ca += ['-a', ILLUMINA_ADAPTER_R1, '-A', ILLUMINA_ADAPTER_R2]
        ca += ['-o', str(o1), '-p', str(o2), str(r1), str(r2)]

        print(f'  [{bn}] cutadapt ...')
        rr = subprocess.run(ca, capture_output=True, text=True)
        if rr.returncode != 0:
            raise RuntimeError(f'cutadapt failed {bn}: {rr.stderr[:1200]}')

    n = len(list(out.glob('*.trim.fastq.gz')))
    print(f'[adapter_trim] DONE: {n} trimmed files')


### Run

Run ERP003950 adapter trimming after reviewing parameters above.


In [ ]:
run_adapter_trim('/data/user/epishkin', 'ERP003950', force=False)
